### **DATA READING**

In [98]:
df_customers = spark.read.format("parquet")\
    .load("abfss://DP700_DEV2@onelake.dfs.fabric.microsoft.com/bronze_lakehouse.Lakehouse/Files/azure_raw_csv/olist_customers_dataset.parquet")

StatementMeta(, 7f5c9159-3d5a-4347-91c4-f93b80ce165a, 100, Finished, Available, Finished, False)

In [99]:
display(df_customers.limit(5))

StatementMeta(, 7f5c9159-3d5a-4347-91c4-f93b80ce165a, 101, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7457075f-db6d-435a-964c-072429d090aa)

## **Data Transformation**

In [100]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

StatementMeta(, 7f5c9159-3d5a-4347-91c4-f93b80ce165a, 102, Finished, Available, Finished, False)

**SELECT**

In [101]:
df_customers = df_customers.select("customer_id", "customer_unique_id", "customer_zip_code_prefix")

StatementMeta(, 7f5c9159-3d5a-4347-91c4-f93b80ce165a, 103, Finished, Available, Finished, False)

In [102]:
display(df_customers.limit(2))

StatementMeta(, 7f5c9159-3d5a-4347-91c4-f93b80ce165a, 104, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b9507f52-345b-4702-a468-7ae97b10b8ee)

**Renaming columns**

In [103]:
df_customers = df_customers.withColumnRenamed("customer_unique_id", "unique_id")\
    .withColumnRenamed("customer_zip_code_prefix", "customer_zip_code")

StatementMeta(, 7f5c9159-3d5a-4347-91c4-f93b80ce165a, 105, Finished, Available, Finished, False)

In [104]:
display(df_customers.limit(10))

StatementMeta(, 7f5c9159-3d5a-4347-91c4-f93b80ce165a, 106, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5dae1d9e-b3fa-4026-b012-df4c9a4e2b0b)

#### **WithColumn Type Casting**

In [105]:
df_customers = df_customers.withColumn("customer_zip_code", col("customer_zip_code").cast(IntegerType()))

StatementMeta(, 7f5c9159-3d5a-4347-91c4-f93b80ce165a, 107, Finished, Available, Finished, False)

In [106]:
display(df_customers.limit(5))

StatementMeta(, 7f5c9159-3d5a-4347-91c4-f93b80ce165a, 108, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f49608fd-23a6-43c5-aa2d-76fab17fedaa)

### **Print Schema**

In [107]:
df_customers.printSchema()

StatementMeta(, 7f5c9159-3d5a-4347-91c4-f93b80ce165a, 109, Finished, Available, Finished, False)

root
 |-- customer_id: string (nullable = true)
 |-- unique_id: string (nullable = true)
 |-- customer_zip_code: integer (nullable = true)



In [108]:
df_orderitems = spark.read.format("parquet")\
    .load("abfss://DP700_DEV2@onelake.dfs.fabric.microsoft.com/bronze_lakehouse.Lakehouse/Files/azure_raw_csv/olist_order_items_dataset.parquet")

StatementMeta(, 7f5c9159-3d5a-4347-91c4-f93b80ce165a, 110, Finished, Available, Finished, False)

In [109]:
display(df_orderitems.limit(10))

StatementMeta(, 7f5c9159-3d5a-4347-91c4-f93b80ce165a, 111, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8b0b41be-703d-4171-931b-01eef92b4a8b)

In [110]:
columns = ["order_item_id", "price", "freight_value"]

for colName in columns:
    df_orderitems = df_orderitems.withColumn(colName, col(colName).cast(FloatType()))


display(df_orderitems.limit(5))

StatementMeta(, 7f5c9159-3d5a-4347-91c4-f93b80ce165a, 112, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0e150375-f7ad-4152-b16a-4b93769a11b6)

### **Timestamp Conversion**

In [111]:
df_orderitems = df_orderitems.withColumn("shipping_limit_date", col("shipping_limit_date").cast(TimestampType()))

StatementMeta(, 7f5c9159-3d5a-4347-91c4-f93b80ce165a, 113, Finished, Available, Finished, False)

### **InferSchema**

In [112]:
df_payments = spark.read.option("inferSchema", True)\
    .parquet("abfss://DP700_DEV2@onelake.dfs.fabric.microsoft.com/bronze_lakehouse.Lakehouse/Files/azure_raw_csv/olist_order_payments_dataset.parquet")
# df now is a Spark DataFrame containing parquet data from "abfss://DP700_DEV2@onelake.dfs.fabric.microsoft.com/bronze_lakehouse.Lakehouse/Files/azure_raw_csv/olist_order_payments_dataset.parquet".
display(df_payments)

StatementMeta(, 7f5c9159-3d5a-4347-91c4-f93b80ce165a, 114, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 909a6b37-08ba-4965-b6d8-b940b22d8cab)

In [113]:
display(df_orderitems.limit(5))

StatementMeta(, 7f5c9159-3d5a-4347-91c4-f93b80ce165a, 115, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b83d0278-9fb8-48ad-a249-508e91325e7a)

In [114]:
df_orderitems = df_orderitems.withColumn("order_item_id", col("order_item_id").cast(IntegerType()))

StatementMeta(, 7f5c9159-3d5a-4347-91c4-f93b80ce165a, 116, Finished, Available, Finished, False)

### **Replace Values**

In [115]:
payment_columns = ["order_id", "payment_sequential", "payment_type", "payment_installments", "payment_value"]

for i in payment_columns:
    if i == "payment_type":
        df_payments = df_payments.withColumn(i, regexp_replace(col(i),"_", " "))
    
    elif (i == "payment_installments") or (i == "payment_value"):
        df_payments = df_payments.withColumn(i,col(i).cast(IntegerType()))


display(df_payments.limit(5))


StatementMeta(, 7f5c9159-3d5a-4347-91c4-f93b80ce165a, 117, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d9f9adf1-abf9-447d-a1bb-bac1087ad36d)

In [116]:
display(df_payments)

StatementMeta(, 7f5c9159-3d5a-4347-91c4-f93b80ce165a, 118, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ec7f14ca-698c-408e-8e05-10f2822a68ca)

In [117]:
df_reviews = spark.read.parquet(\
    "abfss://DP700_DEV2@onelake.dfs.fabric.microsoft.com/bronze_lakehouse.Lakehouse/Files/azure_raw_csv/olist_order_reviews_dataset.parquet")

display(df_reviews.limit(2))

StatementMeta(, 7f5c9159-3d5a-4347-91c4-f93b80ce165a, 119, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 66889665-b9b9-4bca-bf52-195220380794)

### **String Functions**

In [118]:
df_reviews = df_reviews.withColumn("review_comment_title", upper("review_comment_title"))\
    .withColumn("review_comment_message", lower("review_comment_message"))

StatementMeta(, 7f5c9159-3d5a-4347-91c4-f93b80ce165a, 120, Finished, Available, Finished, False)

In [119]:
display(df_reviews.limit(5))

StatementMeta(, 7f5c9159-3d5a-4347-91c4-f93b80ce165a, 121, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fa2fac75-afec-4a20-904c-eaaa5b912ff4)

In [120]:
df_products = spark.read.parquet("abfss://DP700_DEV2@onelake.dfs.fabric.microsoft.com/bronze_lakehouse.Lakehouse/Files/azure_raw_csv/olist_products_dataset.parquet")
# df now is a Spark DataFrame containing parquet data from "abfss://DP700_DEV2@onelake.dfs.fabric.microsoft.com/bronze_lakehouse.Lakehouse/Files/azure_raw_csv/olist_products_dataset.parquet".
display(df_products.limit(5))

StatementMeta(, 7f5c9159-3d5a-4347-91c4-f93b80ce165a, 122, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6313b749-dcf3-42cb-a2a0-72eea6b060c5)

### **Handling NULLS**

In [121]:
df_products = df_products.fillna({"product_category_name": "x", "product_name_lenght": "y", "product_photos_qty": "z"})

display(df_products)

StatementMeta(, 7f5c9159-3d5a-4347-91c4-f93b80ce165a, 123, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e7be1988-56f2-4d01-a48f-713f86eb16d2)

In [122]:
products_columns = ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]
for i in products_columns:
    df_products = df_products.withColumn(i, col(i).cast(IntegerType()))
display(df_products)


StatementMeta(, 7f5c9159-3d5a-4347-91c4-f93b80ce165a, 124, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b78439c2-f277-426c-a106-1ee4cf8dcc68)

**Filter Data**

In [123]:
display(df_products.filter(col("product_weight_g")>300))

StatementMeta(, 7f5c9159-3d5a-4347-91c4-f93b80ce165a, 125, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 369c8458-9039-42a8-b4bb-3faea0fffa87)

In [124]:
display(df_products)

StatementMeta(, 7f5c9159-3d5a-4347-91c4-f93b80ce165a, 126, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 87c3dd89-bc2f-4afe-977e-9ec02d9d40c8)

### **Flaggin - Create a new column**

In [125]:
df_products = df_products.withColumn("300gFlag", when(col("product_weight_g")> 300, "Y").otherwise("N"))

display(df_products)

df = spark.read.parquet("abfss://DP700_DEV2@onelake.dfs.fabric.microsoft.com/bronze_lakehouse.Lakehouse/Files/azure_raw_csv/olist_products_dataset.parquet")
# df now is a Spark DataFrame containing parquet data from "abfss://DP700_DEV2@onelake.dfs.fabric.microsoft.com/bronze_lakehouse.Lakehouse/Files/azure_raw_csv/olist_products_dataset.parquet".
display(df)

StatementMeta(, 7f5c9159-3d5a-4347-91c4-f93b80ce165a, 127, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e8864233-1a9b-4669-b8db-a7d9de7a8b8c)

SynapseWidget(Synapse.DataFrame, 090f679e-ed75-442c-8bd4-b36db2a0c0b7)

In [126]:
df_orders = spark.read.parquet("abfss://DP700_DEV2@onelake.dfs.fabric.microsoft.com/bronze_lakehouse.Lakehouse/Files/azure_raw_csv/olist_orders_dataset.parquet")
# df now is a Spark DataFrame containing parquet data from "abfss://DP700_DEV2@onelake.dfs.fabric.microsoft.com/bronze_lakehouse.Lakehouse/Files/azure_raw_csv/olist_orders_dataset.parquet".
display(df_orders)

StatementMeta(, 7f5c9159-3d5a-4347-91c4-f93b80ce165a, 128, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d9546f95-104a-4db5-8bf4-aef547891fcf)

In [127]:
df_orders = df_orders.dropna(subset= ["order_delivered_carrier_date", "order_delivered_customer_date"])

StatementMeta(, 7f5c9159-3d5a-4347-91c4-f93b80ce165a, 129, Finished, Available, Finished, False)

In [128]:
display(df_orders)

StatementMeta(, 7f5c9159-3d5a-4347-91c4-f93b80ce165a, 130, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7af42dad-a273-4422-9877-528abcd1f350)

In [129]:
df_orders = df_orders.withColumn("order_purchase_timestamp", col("order_purchase_timestamp").cast(TimestampType()))

StatementMeta(, 7f5c9159-3d5a-4347-91c4-f93b80ce165a, 131, Finished, Available, Finished, False)

## **Spark SQL**

In [130]:
df_orders.createTempView("orders_temp_view")

StatementMeta(, 7f5c9159-3d5a-4347-91c4-f93b80ce165a, 132, Finished, Available, Finished, False)

AnalysisException: [TEMP_TABLE_OR_VIEW_ALREADY_EXISTS] Cannot create the temporary view `orders_temp_view` because it already exists.
Choose a different name, drop or replace the existing view,  or add the IF NOT EXISTS clause to tolerate pre-existing views.

In [ ]:
%%sql

SELECT * FROM orders_temp_view

In [ ]:
%%sql

SELECT *,
        row_number() OVER(ORDER BY order_id) as row_number
FROM orders_temp_view


In [90]:
df_sql = spark.sql("""SELECT *,
        row_number() OVER(ORDER BY order_id) as row_number
FROM orders_temp_view""")

StatementMeta(, 7f5c9159-3d5a-4347-91c4-f93b80ce165a, 92, Finished, Available, Finished, False)

In [91]:
display(df_sql)

StatementMeta(, 7f5c9159-3d5a-4347-91c4-f93b80ce165a, 93, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4211fb74-b596-4d40-957d-e1d1941caf3b)

## **Visualization**

In [92]:
display(df_products)

StatementMeta(, 7f5c9159-3d5a-4347-91c4-f93b80ce165a, 94, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5941e4aa-bab5-419d-bfe1-86eb76c24601)

### **Data Writing Scenarios**

#### **Writing Files**

In [93]:
df_sql.write.format("delta")\
    .mode("append")\
    .option("path", "Files/raw_source")\
    .save()

StatementMeta(, 7f5c9159-3d5a-4347-91c4-f93b80ce165a, 95, Finished, Available, Finished, False)

#### **Writing as a managed table**

In [94]:
df_sql.write.format("delta")\
    .mode("append")\
    .saveAsTable("delta_man_tbl")

StatementMeta(, 7f5c9159-3d5a-4347-91c4-f93b80ce165a, 96, Finished, Available, Finished, False)

###### **Write as managed table in my bronze lakehouse**

In [96]:
df_sql.write.format("csv")\
    .mode("append")\
    .saveAsTable("bronze_lakehouse.csv_man_tbl")

StatementMeta(, 7f5c9159-3d5a-4347-91c4-f93b80ce165a, 98, Finished, Available, Finished, False)

#### **Writing as an external table**

In [97]:
df_sql.write.format("delta")\
    .mode("append")\
    .option("path", "Files/my_data")\
    .saveAsTable("ext_table_delta")

StatementMeta(, 7f5c9159-3d5a-4347-91c4-f93b80ce165a, 99, Finished, Available, Finished, False)

In [133]:
df_sql.write.format("delta")\
    .mode("append")\
    .option("path", "abfss://DP700_DEV2@onelake.dfs.fabric.microsoft.com/bronze_lakehouse.Lakehouse/Files/my_data")\
    .saveAsTable("ext_table_delta_ext")

StatementMeta(, 7f5c9159-3d5a-4347-91c4-f93b80ce165a, 135, Finished, Available, Finished, False)

# **DATA WRITING**

In [134]:
files = {
    "enr_customers": df_customers,
    "enr_orderitems": df_orderitems,
    "enr_orders": df_orders,
    "enr_payments": df_payments,
    "enr_reviews": df_reviews,
    "enr_products": df_products
}

for name, df in files.items():
    df.write.format("delta")\
        .mode("append")\
        .saveAsTable(f"Silver_LH.{name}")


StatementMeta(, 7f5c9159-3d5a-4347-91c4-f93b80ce165a, 136, Finished, Available, Finished, False)